# പാഠം 09 - മെറ്റാകോഗ്നിഷൻ ഡിസൈൻ പാറ്റേൺ


## സജ്ജീകരണം

ഈ നോട്ട്‌ബുക്ക് Microsoft Agent Framework ഉപയോഗിച്ച് Metacognition ഡിസൈൻ പാറ്റേൺ പ്രകടിപ്പിക്കുന്നു.

**മുൻ നിബന്ധനകൾ:**
- പരിസ്ഥിതി ചേരുവകൾ മുഖാന്തിരം ക്രമീകരിച്ച Azure OpenAI ഡിപ്പ്ലോയ്മെന്റ്
- Azure CLI പ്രാമാണീകരിച്ചിരിക്കണം (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## മെറ്റാകോഗ്നിഷൻ എന്താണ്?

മെറ്റാകോഗ്നിഷൻ എന്നത് **ചിന്തയെക്കുറിച്ച് ചിന്തിക്കലാണ്**. AI ഏജന്റുകളുടെ സാഹചര്യത്തിൽ, ഇതിന്റെ അർത്ഥം താഴെപ്പറയുന്ന കഴിവുകൾ ഉള്ള ഏജന്റുകൾ രൂപപ്പെടുത്തുകയാണ്:

- **സ്വയം പ്രതിബിംബനം ചെയ്യുക** അവരുടെ സ്വന്തം ഔട്ട്പുട്ടുകളിലും തർക്കപ്രക്രിയയിലും
- **പിശകുകൾ കണ്ടെത്തുക** και സൗമ്യമായി പുനഃസ്ഥാപിക്കുക, മൗനമായി പരാജയപ്പെടുന്ന പകരം
- **മൂല്യനിർണ്ണയം ചെയ്യുക** അവരുടെ പ്രതികരണങ്ങൾ പൂർണവും സഹായകവുമാണോ എന്ന്
- **മാറ്റം വരുത്തുക** അവരുടെ തന്ത്രത്തിൽ ആദ്യ സമീപനം പ്രവർത്തിക്കാത്തപ്പോൾ (ഉദാ., ബാക്കപ്പ് സിസ്റ്റത്തിലേക്ക് മടങ്ങുക)

ഒരു മെറ്റാകോഗ്നിറ്റീവ് ഏജന്റ് വെറുതെ ചോദ്യങ്ങൾക്ക് മാത്രം മറുപടി പറയുന്നവനല്ല — അത് തന്റെ സ്വന്തം പ്രകടനം നിരീക്ഷിക്കുകയും തൽക്ഷണമായി ക്രമീകരിക്കുകയും ചെയ്യുന്നു.


## പ്രാഥമികവും ബാക്കപ്പും ഉപകരണങ്ങളും

ഒരു സാധാരണ മെറ്റാകോഗ്നിഷൻ മാതൃക **ഫോൾബാക്ക് തന്ത്രം** ആണ്. ഏജന്റ് ആദ്യം ഒരു പ്രാഥമിക ഉപകരണം പരീക്ഷിക്കുന്നു; അത് പരാജയപ്പെട്ടാൽ (ഉദാ., 404 error), ഏജന്റ് പരാജയം തിരിച്ചറിഞ്ഞ് സുതാര്യമായി ബാക്കപ്പ് ഉപകരണത്തിലേക്ക് മാറും.

ഇത് യാഥാർത്ഥ്യ ലോകത്തിലെ സിസ്റ്റങ്ങളുമായി സാമ്യമുള്ളതാണ്, അവിടെ പ്രാഥമിക സേവനങ്ങൾ ലഭ്യമല്ലാതിരിക്കാം, അതിനാൽ ഏജന്റുകൾക്ക് മറ്റൊരു മാർഗം തിരഞ്ഞെടുക്കുന്നതിന് മുമ്പ് പ്രശ്നം സ്വയം പരിശോധന നടത്തുകയും തിരിച്ചറിഞ്ഞുകൂടിയിരിക്കണം.

താഴെ നാം രണ്ട് ഫ്ലൈറ്റ് ലുക്ക്‌അപ്പ് ഉപകരണങ്ങൾ നിർവചിക്കുന്നു:
- **പ്രാഥമികം** — പാരിസ്, ടോക്കിയോ, ബാർസലോണ എന്നിവയെ ഉൾക്കൊള്ളുന്നു
- **ബാക്കപ്പ്** — ബർലിൻ, സിഡ്നി, ന്യൂയോർക്ക് സിറ്റി എന്നിവയെ ഉൾക്കൊള്ളുന്നു


In [ ]:
@tool(approval_mode="never_require")
def get_flight_times(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times for a destination (primary source)."""
    flights = {
        "Paris": "Departures: 08:00, 12:30, 17:45 — from $350",
        "Tokyo": "Departures: 11:00, 23:30 — from $890",
        "Barcelona": "Departures: 07:15, 14:00, 19:30 — from $280",
    }
    if destination in flights:
        return flights[destination]
    raise Exception(f"404: No flights found for {destination} in primary system")


@tool(approval_mode="never_require")
def get_flight_times_backup(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times from backup system (used when primary fails)."""
    backup_flights = {
        "Berlin": "Departures: 09:00, 16:00 — from $220",
        "Sydney": "Departures: 22:00 — from $1200",
        "New York City": "Departures: 06:00, 10:30, 15:00, 20:00 — from $450",
    }
    return backup_flights.get(
        destination,
        f"No flights found for {destination} in any system. Please try again later.",
    )

## പിശക് വീണ്ടെടുക്കലോടുകൂടിയ സ്വയം-പരിശോധിക്കുന്ന ഏജന്റ്

താഴെ കാണുന്ന ഏജന്റിനെ പ്രാഥമിക ഫ്ലൈറ്റ് സിസ്റ്റം ആദ്യം പരീക്ഷിക്കാൻ, പരാജയങ്ങൾ തിരിച്ചറിയാൻ, കൂടാതെ സുതാര്യമായി ബാക്കപ്പ് സിസ്റ്റത്തിലേക്ക് തിരിയാൻ നിർദ്ദേശിച്ചിരിക്കുന്നു. ഓരോ പ്രതികരണത്തിനുശേഷവും അത് സംക്ഷിപ്തമായി സ്വയംമൂല്യനിർണയം നടത്തുന്നു, ഉപയോക്താവിന്റെ ചോദ്യം പൂർണ്ണമായി ഉത്തരം നൽകിയോ എന്ന്.


In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="FlightBookingAgent",
    instructions="""You are a flight booking agent with self-reflection capabilities.

When looking up flights:
1. Try the primary flight system first (get_flight_times)
2. If the primary system fails (404 error), acknowledge the error and try the backup system (get_flight_times_backup)
3. Always explain to the user what happened — be transparent about fallbacks
4. If both systems fail, apologize and suggest alternatives

After each response, briefly evaluate whether your answer was complete and helpful.""",
)

# Test with a destination in primary system
print("=== Test 1: Destination in primary system ===")
response = await agent.run(
    "What flights are available to Paris?",
    )
print(response)

# Test with a destination only in backup system
print("\n=== Test 2: Destination only in backup system ===")
response = await agent.run(
    "What flights are available to Berlin?",
    )
print(response)

## സ്വയം-മൂല്യനിർണയ മാതൃക

മേറ്റാകോഗ്നിഷന്റെ മറ്റൊരു ദിശയാണ് **സ്വയം-മൂല്യനിർണയം**: ഒരു വേർതിരിച്ചായ ഏജന്റോ (അല്ലെങ്കിൽ രണ്ടാമത് റണ്ണിൽ അതേ ഏജന്റോവോ) ഒരു പ്രതികരണം പൂര്‍ത്തിത്ത്വത്തിലും, കൃത്യതയിലും, ഉപകാരപ്രദതയിലും പരിശോധിക്കുന്നു.

താഴെ നാം `ResponseEvaluator` ഏജന്റിനെ സൃഷ്ടിക്കുന്നു, അത് യാത്രാ-ഏജന്റിന്റെ പ്രതികരണങ്ങളെ മൂന്ന്维ങ്ങളിൽ സ്കോർ ചെയ്യുന്നു.


In [ ]:
evaluation_agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="ResponseEvaluator",
    instructions="""You are a quality evaluator for travel agent responses.
Given a travel question and the agent's response, evaluate:
1. Completeness: Did it answer all parts of the question? (1-5)
2. Accuracy: Is the information correct? (1-5)
3. Helpfulness: Would a traveler find this useful? (1-5)
Provide a brief evaluation with scores and one suggestion for improvement.""",
)

# Evaluate the agent's response from Test 1
eval_prompt = f"""Question: What flights are available to Paris?
Agent Response: {response}

Please evaluate the above response."""

evaluation = await evaluation_agent.run(eval_prompt)
print("=== Self-Evaluation ===")
print(evaluation)

## സംഗ്രഹം

ഈ പാഠത്തിൽ നിങ്ങൾ Microsoft Agent Framework ഉപയോഗിച്ച് **മെറ്റകോഗ്നിറ്റീവ് ഏജന്റുകൾ** എങ്ങനെ നിർമ്മിക്കാമെന്ന് പഠിച്ചു:

- **സ്വയംപരിശോധന**: തങ്ങളുടെ തന്നെ യുക്തിവിചിന്തനകൾ നിരീക്ഷിച്ച് സംഭവിച്ചത് എന്താണെന്ന് വ്യക്തതയോടെ അറിയിക്കുന്ന ഏജന്റുകൾ.
- **ഫാൽബാക്കുകളോടെയുള്ള പിശക് വീണ്ടെടുക്കൽ**: ഒരു പ്രധാന + ബാക്കപ്പ് ടൂൾ പാറ്റേൺ, ഏജന്റ് പരാജയങ്ങൾ കണ്ടെത്തുമ്പോൾ (ഉദാ., 404 പിശകുകൾ) സ്വയமേവ മറ്റൊരു ഉറവിടം പരീക്ഷിക്കുന്നത്.
- **സ്വയംമൂല്യനിർണ്ണയം**: പൂര്ണ്ണത, കൃത്യത, സഹായകരത എന്നിവയുടെ അടിസ്ഥാനത്തിൽ മറുപടികൾക്ക് സ്കോർ നൽകുന്നത് പോലെ വേറിട്ട ഒരു മൂല്യനിർണയ ഏജന്റ്.

ഈ മാതൃകകൾ ഏജന്റുകളെ കൂടുതൽ ദൃഢവും, വ്യക്തതയുള്ളതുമായ, വിശ്വാസയോഗ്യവുമായതാക്കുന്നു — പ്രൊഡക്ഷൻ വിന്യാസങ്ങൾക്ക് നിർണായകഗുണങ്ങൾ.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
അസ്വീകരണം:
ഈ ഡോക്യുമെന്റ് AI വിവർത്തന സേവനമായ Co-op Translator (https://github.com/Azure/co-op-translator) ഉപയോഗിച്ച് വിവർത്തനം ചെയ്തതാണ്. ഞങ്ങൾ കൃത്യതയ്ക്കായി ശ്രമിച്ചിരുന്നെങ്കിലും, ഓട്ടോമേറ്റഡ് വിവർത്തനങ്ങളിൽ പിഴവുകളോ അസാമാന്യതകളോ ഉണ്ടാകാമെന്നത് ദയവായി ശ്രദ്ധിക്കുക. മാതൃഭാഷയിലുള്ള യഥാർത്ഥ രേഖയാണ് ഔദ്യോഗിക ഉറവിടമായി കണക്കാക്കേണ്ടത്. നിർണായകമായ വിവരങ്ങൾക്ക് പ്രൊഫഷണൽ മനുഷ്യവിവർത്തനം ശുപാർശ ചെയ്യുന്നു. ഈ വിവർത്തനം ഉപയോഗിച്ചതിലൂടെ ഉണ്ടാകുന്ന ഏതെങ്കിലും തെറ്റിദ്ധാരണകൾക്കും തെറ്റായ വ്യാഖ്യാനങ്ങൾക്കും ഞങ്ങൾ ഉത്തരവാദികളല്ല.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
